In [10]:
import pandas as pd

# Load the dataset
df = pd.read_csv('dirty_financial_transactions.csv')

# First look
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

(100000, 8)
  Transaction_ID Transaction_Date Customer_ID    Product_Name  Quantity  \
0          T0001       2024-08-02       C2205      Headphones      -5.0   
1          T0002       2020-02-10       C3156         Coffee      469.0   
2          T0003       2025-02-30       C2919          Tablet      -4.0   
3          T0004       2020-08-17       C3009             Tab      -7.0   
4          T0005       2025-02-30       C3488  Coffee Machine     -10.0   

                 Price Payment_Method Transaction_Status  
0              $420.21        pay pal                NaN  
1  -445.34202525395585     creditcard            Pending  
2    810.9930123946459    credit card          completed  
3    868.6083413217348         PayPal            Pending  
4   -763.1224490039416         PayPal          completed  
Transaction_ID         object
Transaction_Date       object
Customer_ID            object
Product_Name           object
Quantity              float64
Price                  object
Pay

In [11]:
# Check how many duplicates exist before dropping
print("Rows before:", len(df))
print("Duplicate rows:", df.duplicated().sum())

# Drop duplicate rows, keep the first occurrence
df = df.drop_duplicates()

# Confirm they're gone
print("Rows after:", len(df))

Rows before: 100000
Duplicate rows: 994
Rows after: 99006


In [12]:
# Strip $ sign AND any leading/trailing whitespace
df['Price'] = df['Price'].str.replace('$', '', regex=False).str.strip()
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Convert negatives to positive instead of dropping them
df['Price'] = df['Price'].abs()

# Confirm
print("Rows remaining:", len(df))
print(df['Price'].describe())
print("Nulls in Price:", df['Price'].isnull().sum())

Rows remaining: 99006
count    65875.000000
mean       525.209593
std        274.168334
min         50.028806
25%        287.874728
50%        525.379269
75%        762.547209
max        999.987698
Name: Price, dtype: float64
Nulls in Price: 33131


In [13]:
# Check before
print("Before:")
print(df['Quantity'].describe())
print("Nulls in Quantity:", df['Quantity'].isnull().sum())

# Convert negatives to positive
df['Quantity'] = df['Quantity'].abs()

# Confirm
print("\nAfter:")
print(df['Quantity'].describe())
print("Nulls in Quantity:", df['Quantity'].isnull().sum())

Before:
count    94034.000000
mean       183.813450
std        299.213589
min        -10.000000
25%         -3.000000
50%          6.000000
75%        327.000000
max       1000.000000
Name: Quantity, dtype: float64
Nulls in Quantity: 4972

After:
count    94034.000000
mean       187.486930
std        296.925568
min          1.000000
25%          4.000000
50%          8.000000
75%        327.000000
max       1000.000000
Name: Quantity, dtype: float64
Nulls in Quantity: 4972


In [14]:
# Check before
print("Before:")
print(df['Payment_Method'].value_counts())

# Standardise — strip whitespace first, then map to clean values
df['Payment_Method'] = df['Payment_Method'].str.strip()

df['Payment_Method'] = df['Payment_Method'].replace({
    'pay pal'    : 'PayPal',
    'PayPal'     : 'PayPal',
    'creditcard' : 'Credit Card',
    'credit card': 'Credit Card',
    'Credit Card': 'Credit Card',
    'Cash'       : 'Cash'
})

# Confirm
print("\nAfter:")
print(df['Payment_Method'].value_counts())

Before:
Payment_Method
Credit Card    14290
creditcard     14233
pay pal        14219
PayPal         14132
credit card    14081
Cash           14053
PayPal         13998
Name: count, dtype: int64

After:
Payment_Method
Credit Card    42604
PayPal         42349
Cash           14053
Name: count, dtype: int64


In [15]:
# Check before
print("Before:")
print(df['Transaction_Status'].value_counts())

# Standardise
df['Transaction_Status'] = df['Transaction_Status'].str.strip().str.lower()

df['Transaction_Status'] = df['Transaction_Status'].replace({
    'completed' : 'Completed',
    'complete'  : 'Completed',
    'pending'   : 'Pending',
    'failed'    : 'Failed'
})

# Confirm
print("\nAfter:")
print(df['Transaction_Status'].value_counts())
print("Nulls:", df['Transaction_Status'].isnull().sum())

Before:
Transaction_Status
Failed       16629
complete     16541
completed    16500
Pending      16443
Completed    16372
Name: count, dtype: int64

After:
Transaction_Status
Completed    49413
Failed       16629
Pending      16443
Name: count, dtype: int64
Nulls: 16521


In [18]:
# Check unique non-null values that are now NaT
raw = pd.read_csv('dirty_financial_transactions.csv')['Transaction_Date']
parsed = pd.to_datetime(raw, errors='coerce')

invalid_mask = parsed.isnull() & raw.notnull()

print("Unparseable date count:", invalid_mask.sum())
print("\nSample invalid values:")
print(raw[invalid_mask].value_counts().head(20))

Unparseable date count: 63381

Sample invalid values:
Transaction_Date
2023-13-01    31834
2025-02-30    31547
Name: count, dtype: int64


In [19]:
# Convert to datetime — invalid dates (month 13, Feb 30) become NaT
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

# Confirm
print(df['Transaction_Date'].dtype)
print("Nulls:", df['Transaction_Date'].isnull().sum())
print(df['Transaction_Date'].describe())

datetime64[ns]
Nulls: 67566
count                            31440
mean     2022-07-23 21:22:43.053434624
min                2020-01-01 00:00:00
25%                2021-04-17 00:00:00
50%                2022-07-26 00:00:00
75%                2023-11-01 00:00:00
max                2025-02-01 00:00:00
Name: Transaction_Date, dtype: object


In [20]:
# Remove extra spaces — catches "T0001 " or " T0001" hiding in the data
df['Transaction_ID'] = df['Transaction_ID'].astype(str).str.strip()

# Extract only the numeric part — pulls "0001" from "T0001"
df['Transaction_ID'] = df['Transaction_ID'].str.extract(r'(\d+)')[0]

# Convert to numeric — prepares for sequence filling
df['Transaction_ID'] = pd.to_numeric(df['Transaction_ID'], errors='coerce')

# Reassign a clean sequential range — ignores whatever was there, guarantees uniqueness
df['Transaction_ID'] = range(1, len(df) + 1)

# Reattach the T prefix with zero-padding — T1 becomes T0001
df['Transaction_ID'] = 'T' + df['Transaction_ID'].astype(str).str.zfill(4)

df.head(10)

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,5.0,420.210000,PayPal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,445.342025,Credit Card,Pending
2,T0003,NaT,C2919,Tablet,4.0,810.993012,Credit Card,Completed
3,T0004,2020-08-17,C3009,Tab,7.0,868.608341,PayPal,Pending
4,T0005,NaT,C3488,Coffee Machine,10.0,763.122449,PayPal,Completed
5,T0006,2021-10-26,C4241,Smartphone,598.0,NaN,PayPal,Completed
6,T0007,NaT,C1313,Laptop,10.0,NaN,Credit Card,Completed
7,T0008,NaT,C4736,Headphones,669.0,86.921269,Cash,NaN
8,T0009,NaT,C3387,Tablet,10.0,461.701984,PayPal,NaN
9,T0010,NaT,C2846,Laptop,1.0,404.890707,Credit Card,Pending


In [21]:
print("Nulls before:")
print(df.isnull().sum())

# Numeric — fill with median
df['Price'] = df['Price'].fillna(df['Price'].median())
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# Categorical — fill with mode
df['Transaction_Status'] = df['Transaction_Status'].fillna(df['Transaction_Status'].mode()[0])
df['Customer_ID'] = df['Customer_ID'].fillna(df['Customer_ID'].mode()[0])

print("\nNulls after:")
print(df.isnull().sum())

Nulls before:
Transaction_ID            0
Transaction_Date      67566
Customer_ID            4830
Product_Name              0
Quantity               4972
Price                 33131
Payment_Method            0
Transaction_Status    16521
dtype: int64

Nulls after:
Transaction_ID            0
Transaction_Date      67566
Customer_ID               0
Product_Name              0
Quantity                  0
Price                     0
Payment_Method            0
Transaction_Status        0
dtype: int64


In [22]:
# Checking the Unique name
df['Product_Name'].unique()
# df['Product_Name'].value_counts()

array(['Headphones', 'Coffee ', 'Tablet', 'Tab', 'Coffee Machine',
       'Smartphone', 'Laptop', 'Coffee Ma', 'Cof', 'Smar', 'Coffee M',
       'T', 'Smartp', 'Headp', 'Smart', 'La', 'Lapt', 'Tabl', 'L', 'C',
       'Smartph', 'Hea', 'Head', 'Smartpho', 'Lapto', 'Headphon', 'Table',
       'Co', 'Headphone', 'Coffee Mac', 'Sm', 'Coffee', 'Headph', 'S',
       'Coffee Mach', 'Smartphon', 'Headpho', 'Coffee Machin', 'Coff',
       'Lap', 'H', 'He', 'Ta', 'Coffee Machi', 'Coffe', 'Sma'],
      dtype=object)

In [23]:
#Lowercase and strip the column temporarily
df['Product_Name_temp'] = df['Product_Name'].str.lower().str.strip()

# replacing the wrong product name with original
replace_name = {
    'tab' : 'Tablet',
    'coffee ma' : 'Coffee Machine',
    'coffee' : 'Coffee Machine',
    'cof' : 'Coffee Machine',
    'smar' : 'Smartphone',
    'coffee m' : 'Coffee Machine',
    't' : 'Tablet',
    'smartpho' : 'Smartphone',
    'headp' : 'Headphones',
    'smart' : 'Smartphone',
    'smartph':'Smartphone',
    'la' : 'Laptop',
    'lapt' : 'Laptop',
    'tabl' : 'Tablet',
    'l' : 'Laptop',
    'c' : 'Coffee Machine',
    'co' : 'Coffee Machine',
    'headphone': 'Headphones',
    'coffee mac' : 'Coffee Machine',
    'sm' : 'Smartphone',
    'headph' : 'Headphones',
    's' : 'Smartphone',
    'coffee mach' : 'Coffee Machine',
    'smartphon' : 'Smartphone',
    'headpho' : 'Headphones',
    'coffee machin' : 'Coffee Machine',
    'coff' : 'Coffee Machine',
    'lap' : 'Laptop',
    'h' : 'Headphones',
    'he' : 'Headphones',
    'ta': 'Tablet',
    'coffee machi' : 'Coffee Machine',
    'coffe' : 'Coffee Machine',
    'sma' : 'Smartphone',
    'smartp' : 'Smartphone',
    'hea' : 'Headphones',
    'headphon': 'Headphones',
    'head' : 'Headphones',
    'lapto' :  'Laptop',
    'table' : 'Tablet'
}

# replacing using lowercase column
df['Product_Name_temp'] = df['Product_Name_temp'].replace(replace_name)

# Converting back to capital latter
df['Product_Name'] = df['Product_Name_temp'].str.title()

#Drop temporary column
df.drop(columns=['Product_Name_temp'], inplace=True)

df['Product_Name'].value_counts()

Product_Name
Tablet            19964
Smartphone        19868
Laptop            19770
Headphones        19731
Coffee Machine    19673
Name: count, dtype: int64

In [25]:
from sqlalchemy import create_engine

username = "root"
password = "Mastermind001"
host = "127.0.0.1"
port = "3306"
database = "financial_transactions_db"  # ← new database

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Export
table_name = "financial_transactions"
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Verify
pd.read_sql("SELECT * FROM financial_transactions LIMIT 5;", engine)

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,5.0,420.210000,PayPal,Completed
1,T0002,2020-02-10,C3156,Coffee Machine,469.0,445.342025,Credit Card,Pending
2,T0003,NaT,C2919,Tablet,4.0,810.993012,Credit Card,Completed
3,T0004,2020-08-17,C3009,Tablet,7.0,868.608341,PayPal,Pending
4,T0005,NaT,C3488,Coffee Machine,10.0,763.122449,PayPal,Completed
